# 23 · K-Means Clustering
> **Part:** Part 4 · Unsupervised learning  
> **Family:** `Unsupervised · Clustering (centroid-based)`

## Definition
K-means partitions n points into k clusters by repeatedly (1) assigning each point to its nearest **centroid** and (2) moving each centroid to the mean of the points
assigned to it (Lloyd's algorithm). It minimizes the within-cluster sum of squares (inertia).

## Intuition
Place k pins on the map at random. Every customer goes to the nearest pin; then move each pin to the centre of its customers; the borders (Voronoi cells) shift, so repeat.
It always converges, but maybe to a *local* optimum, so initialisation matters (k-means++ spreads the starting pins).


## The Math

```text
assign:  cᵢ = argmin_k ||xᵢ − μ_k||²              update:  μ_k = mean{ xᵢ : cᵢ = k }
    objective (inertia): J = Σᵢ ||xᵢ − μ_{cᵢ}||²   (never increases; guaranteed to converge)
    Choosing k: elbow method (where J stops dropping fast), silhouette score, gap statistic, or business needs.
    k-means++: first centre uniformly at random, each next centre sampled with probability ∝ D(x)² (distance to nearest chosen centre).
```


In [ ]:
# Setup, Imports & Styling
%matplotlib inline
import os
import pathlib
import sys
from IPython.display import Image, display

# Ensure parent folder is in path to import mlatlas
parent_dir = pathlib.Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlatlas import data, viz, cli

# Ensure interactive inline mode and configure Seaborn theme
plt.ion()
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})


### Step 1: Unlabeled Cluster Distribution Visualization
We visualize unlabeled mall customer spending data exhibiting clear spatial clusters.


In [ ]:
X, _ = data.mall_customers(n=300, seed=0)

plt.figure(figsize=(8, 4.5))
sns.scatterplot(x=X[:, 0], y=X[:, 1], color="#3b6fb6", s=50, alpha=0.8, edgecolor="w")
plt.title("Mall Customers: Annual Income vs. Spending Score", fontsize=12, fontweight="bold")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.tight_layout()
plt.show()


### Step 2: From-Scratch Algorithm Implementation
Below is the complete implementation of the model components in pure NumPy.


In [ ]:
# Helper Classes & Mathematical Logic
def assign(X, C): return ((X[:, None] - C[None]) ** 2).sum(-1).argmin(1)
def inertia(X, C, l): return ((X - C[l]) ** 2).sum()
def lloyd(X, C, iters=30):
    hist = [(C.copy(), assign(X, C), "init")]
    for _ in range(iters):
        l = assign(X, C); hist.append((C.copy(), l, "assign")); Cn = np.array([X[l == k].mean(0) if (l == k).any() else C[k] for k in range(len(C))]); hist.append((Cn.copy(), l, "update"))
        if np.allclose(Cn, C): break
        C = Cn
    return hist
def kpp(X, k, rng):
    C = [X[rng.integers(len(X))]]
    for _ in range(k - 1): d = ((X[:, None] - np.array(C)[None]) ** 2).sum(-1).min(1); C.append(X[rng.choice(len(X), p=d / d.sum())])
    return np.array(C)


In [ ]:
# Main Algorithm & Training Routine
def main(gif, quick):
    X, _ = data.mall_customers(300, seed=1); rng = np.random.default_rng(4); k = 5
    el = [min(inertia(X, h[0], h[1]) for h in [lloyd(X, X[rng.choice(len(X), kk, replace=False)])[-1]] for _ in range(3)) for kk in range(1, 9)]
    print("  elbow (inertia for k=1..8):", [int(e) for e in el])
    bad = np.array([X[np.argmin(X[:, 0] + X[:, 1])], X[np.argmin(X[:, 0] + X[:, 1])] + [3, 4], X[np.argmax(X[:, 0] + X[:, 1])], X[np.argmax(X[:, 0] + X[:, 1])] - [3, 4], X[np.argmax(X[:, 0])]])
    H = lloyd(X, bad); Hp = lloyd(X, kpp(X, k, np.random.default_rng(1))); print(f"  bad start final inertia {inertia(X, H[-1][0], H[-1][1]):.0f} vs k-means++ {inertia(X, Hp[-1][0], Hp[-1][1]):.0f}  ({len(H) // 2} vs {len(Hp) // 2} iterations)")
    J = [inertia(X, c, l) for c, l, _ in H]; fig, (a1, a2) = viz.subplots(1, 2, w=11, h=4.5)

    def update(i):
        a1.clear(); a2.clear(); C, l, ph = H[i]; viz.scatter(a1, X, l, s=18)
        for kk in range(k): a1.scatter(*C[kk], marker="X", s=190, color=viz.PALETTE[kk], edgecolor="k", linewidth=1.5, zorder=6)
        if ph == "update":
            for kk in range(k): a1.annotate("", C[kk], H[i - 1][0][kk], arrowprops=dict(arrowstyle="->", lw=1.5))
        a1.set(title=f"iteration {(i + 1) // 2}: {ph.upper()} step", xlabel="annual income (k$)", ylabel="spending score", xlim=(0, 105), ylim=(0, 105))
        a2.plot(J[:i + 1], "o-", color=viz.PALETTE[3], ms=3); a2.set(xlim=(0, len(H)), ylim=(0, max(J) * 1.05), title="Inertia (within-cluster sum of squares)", xlabel="half-step")
    viz.animate(fig, update, range(len(H) if not quick else min(len(H), 12)), gif, fps=2)


### Step 3: Voronoi Partition & Centroid Convergence
We fit K-Means ($k=5$) and plot the resulting Voronoi partitions along with the final converged centroid coordinates.


In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=5, init="k-means++", n_init=10, random_state=42).fit(X)
labels = km.labels_
centers = km.cluster_centers_

plt.figure(figsize=(9, 5))
x_min, x_max = X[:, 0].min() - 5, X[:, 0].max() + 5
y_min, y_max = X[:, 1].min() - 5, X[:, 1].max() + 5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))
Z = km.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.25, cmap="Set2")
sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=labels, palette="Set2", s=50, edgecolor="k")
plt.scatter(centers[:, 0], centers[:, 1], marker="X", s=220, color="red", edgecolor="black", label="Centroids")
plt.title(f"K-Means Customer Segmentation (k=5, Inertia: {km.inertia_:.1f})", fontsize=12, fontweight="bold")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend()
plt.tight_layout()
plt.show()


### Step 4: Industry Benchmark (Elbow Method & Silhouette Score)
We calculate inertia across $k=2..8$ to inspect the elbow curve.


In [ ]:
from sklearn.metrics import silhouette_score

inertias = [KMeans(k, n_init=5, random_state=42).fit(X).inertia_ for k in range(2, 8)]
sil = silhouette_score(X, labels)
print("Inertias (k=2 to 7):", [round(i, 1) for i in inertias])
print(f"Silhouette Score (k=5): {sil:.4f}")


### Step 5: Visualizing Learning Dynamics

Mall customers (annual income vs spending score) with a deliberately bad random start. Each iteration has two frames: **assign** (points recolour) then **update** (centroids jump to the
cluster means, arrows show the move). Right: inertia plummets then flat-lines. The console prints the elbow curve and compares k-means++.


In [ ]:
# Display the learning animation inline
media_file = f"../outputs/23_k_means.gif"
if os.path.exists(media_file):
    display(Image(filename=media_file))
elif os.path.exists(f"outputs/23_k_means.gif"):
    display(Image(filename=f"outputs/23_k_means.gif"))


## Practical tips

- Scale features; assumes round, similarly sized clusters; sensitive to outliers (use k-medoids / GMM / DBSCAN otherwise).
- Use for customer segmentation, colour quantization, vector quantization, initialising other models.


## 🧠 Check Your Understanding

<details>
<summary><b>1. Why does K-Means struggle with elongated or non-convex clusters?</b></summary>
Because K-Means uses Euclidean distance from a central point, implicitly assuming spherical clusters of equal variance. Points are assigned based on perpendicular bisectors, producing convex Voronoi cells.
</details>

<details>
<summary><b>2. How does k-means++ initialization prevent poor local optima?</b></summary>
Standard random initialization can place multiple centroids in the same cluster. K-means++ chooses the first centroid at random, then selects subsequent centroids with probability proportional to the squared distance from the nearest existing centroid.
</details>


## 🧪 Guided Experiments

- Set k=3 and k=8 and look at the segments.
- Run 20 random restarts and keep the best inertia.
